# DINO on STL-10 → CIFAR-10

Teacher–student with softmax CE, centering + sharpening anti-collapse.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import math
import os

DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "DATA"))
CKPT_DIR = './checkpoints/9_DINO'
os.makedirs(CKPT_DIR, exist_ok=True)

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device = {device}")

---
## Dataset

In [ ]:
BATCH_SIZE = 128
IMG_SIZE   = 32
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2470, 0.2435, 0.2616)


class TwoViews:
    def __init__(self, transform):
        self.transform = transform
    def __call__(self, x):
        return self.transform(x), self.transform(x)


color_jitter = transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)
ssl_aug = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([color_jitter], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
train_probe_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

pretrain_set = torchvision.datasets.STL10(root=DATA_DIR, split='unlabeled', download=True,
                                           transform=TwoViews(ssl_aug))
probe_train  = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=train_probe_tf)
probe_test   = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=test_tf)

pretrain_loader     = DataLoader(pretrain_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, drop_last=True)
probe_train_loader  = DataLoader(probe_train,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
probe_test_loader   = DataLoader(probe_test,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

sample_img, _ = probe_train[0]
CHANNELS, H, W = sample_img.shape
NUM_CLASSES    = len(probe_train.classes)
print(f"pretrain {len(pretrain_set)} | probe train {len(probe_train)} | probe test {len(probe_test)}")

---
## Hyperparameters

In [ ]:
PATCH_SIZE  = 4
GRID_SIZE   = IMG_SIZE // PATCH_SIZE
N_PATCHES   = GRID_SIZE ** 2
PATCH_DIM   = PATCH_SIZE ** 2 * CHANNELS

D_MODEL    = 128
NUM_HEADS  = 4
NUM_LAYERS = 6
D_FF       = 4 * D_MODEL
DROPOUT    = 0.1

PRETRAIN_EPOCHS = 20
PROBE_EPOCHS    = 30
FT_EPOCHS       = 30

PROJ_DIM = 256
TAU_S = 0.1
TAU_T = 0.04
CENTER_MOM = 0.9
TAU_EMA = 0.996

---
## Architecture

In [ ]:
def patchify(images, patch_size):
    B, C, H, W = images.shape
    P = patch_size
    x = images.reshape(B, C, H // P, P, W // P, P)
    x = x.permute(0, 2, 4, 3, 5, 1)
    x = x.reshape(B, (H // P) * (W // P), P * P * C)
    return x

In [ ]:
class ViTEncoder(nn.Module):
    """ViT-Tiny backbone — outputs (B, D) CLS feature."""
    def __init__(self, img_size=IMG_SIZE, patch_size=PATCH_SIZE, channels=CHANNELS,
                 d_model=D_MODEL, num_heads=NUM_HEADS, num_layers=NUM_LAYERS,
                 d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        n_patches = (img_size // patch_size) ** 2
        patch_dim = patch_size ** 2 * channels
        self.patch_size = patch_size

        self.projection    = nn.Linear(patch_dim, d_model)
        self.cls_token     = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embedding = nn.Parameter(torch.randn(1, n_patches + 1, d_model) * 0.02)
        self.dropout       = nn.Dropout(dropout)

        block = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=d_ff,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(block, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, images):
        B = images.shape[0]
        x = patchify(images, self.patch_size)
        x = self.projection(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embedding
        x = self.dropout(x)
        x = self.transformer(x)
        x = self.norm(x)
        return x[:, 0, :]

In [ ]:
class DINOHead(nn.Module):
    def __init__(self, in_dim=D_MODEL, hidden=D_MODEL*2, out_dim=PROJ_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.GELU(),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x): return self.net(x)


class DINO(nn.Module):
    def __init__(self):
        super().__init__()
        self.student_enc  = ViTEncoder(); self.student_head  = DINOHead()
        self.teacher_enc  = ViTEncoder(); self.teacher_head  = DINOHead()
        for ps, pt in zip(self.student_enc.parameters(), self.teacher_enc.parameters()):
            pt.data.copy_(ps.data); pt.requires_grad = False
        for ps, pt in zip(self.student_head.parameters(), self.teacher_head.parameters()):
            pt.data.copy_(ps.data); pt.requires_grad = False
        self.register_buffer('center', torch.zeros(1, PROJ_DIM))

    @torch.no_grad()
    def _update_teacher(self, tau):
        for ps, pt in zip(self.student_enc.parameters(), self.teacher_enc.parameters()):
            pt.data.mul_(tau).add_(ps.data, alpha=1.0 - tau)
        for ps, pt in zip(self.student_head.parameters(), self.teacher_head.parameters()):
            pt.data.mul_(tau).add_(ps.data, alpha=1.0 - tau)

    @torch.no_grad()
    def _update_center(self, teacher_out):
        batch_center = teacher_out.mean(dim=0, keepdim=True)
        self.center.mul_(CENTER_MOM).add_(batch_center, alpha=1.0 - CENTER_MOM)

    def forward(self, v1, v2):
        s1 = self.student_head(self.student_enc(v1))
        s2 = self.student_head(self.student_enc(v2))
        with torch.no_grad():
            t1 = self.teacher_head(self.teacher_enc(v1))
            t2 = self.teacher_head(self.teacher_enc(v2))
        # teacher distributions (centered + sharpened)
        pt1 = F.softmax((t1 - self.center) / TAU_T, dim=-1)
        pt2 = F.softmax((t2 - self.center) / TAU_T, dim=-1)
        # student log-distributions
        ls1 = F.log_softmax(s1 / TAU_S, dim=-1)
        ls2 = F.log_softmax(s2 / TAU_S, dim=-1)
        # cross-views CE
        loss = (-(pt2.detach() * ls1).sum(dim=-1).mean()
                -(pt1.detach() * ls2).sum(dim=-1).mean()) / 2
        # updates
        self._update_center(torch.cat([t1, t2], dim=0))
        self._update_teacher(TAU_EMA)
        return loss


dino = DINO().to(device)
trainable = list(dino.student_enc.parameters()) + list(dino.student_head.parameters())
print(f"trainable params : {sum(p.numel() for p in trainable):,}")

---
## Pre-training

In [ ]:
optimizer = torch.optim.AdamW(trainable, lr=3e-4, weight_decay=1e-4)
pretrain_losses = []
for epoch in range(1, PRETRAIN_EPOCHS + 1):
    dino.train()
    total, n = 0.0, 0
    for (v1, v2), _ in pretrain_loader:
        v1, v2 = v1.to(device), v2.to(device)
        loss = dino(v1, v2)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * v1.size(0); n += v1.size(0)
    avg = total / n
    pretrain_losses.append(avg)
    print(f"pretrain {epoch:2d}/{PRETRAIN_EPOCHS} | loss {avg:.4f}")
    torch.save({'epoch': epoch, 'model': dino.state_dict(), 'losses': pretrain_losses},
               os.path.join(CKPT_DIR, 'dino_pretrain.pt'))

plt.figure(figsize=(6, 3.5)); plt.plot(pretrain_losses); plt.title('DINO pre-training loss')
plt.xlabel('epoch'); plt.ylabel('CE'); plt.grid(True, alpha=0.3); plt.show()

---
## Encoder wrapper

In [ ]:
def build_eval_encoder():
    m = DINO().to(device)
    ckpt = torch.load(os.path.join(CKPT_DIR, 'dino_pretrain.pt'), map_location=device)
    m.load_state_dict(ckpt['model'])
    return m.student_enc

---
## Linear probe

In [ ]:
class LinearProbe(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.head    = nn.Linear(D_MODEL, num_classes)
        for p in self.encoder.parameters():
            p.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            h = self.encoder(x)
        return self.head(h)

In [ ]:
probe = LinearProbe(build_eval_encoder(), NUM_CLASSES).to(device)
print(f"trainable : {sum(p.numel() for p in probe.head.parameters()):,}")

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += (logits.argmax(dim=1) == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)
        correct    += (logits.argmax(dim=1) == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total

In [ ]:
optimizer = torch.optim.AdamW(probe.head.parameters(), lr=0.001, weight_decay=0.0 if True else 0.05)
criterion = nn.CrossEntropyLoss()

train_losses, test_losses = [], []
train_accs,   test_accs   = [], []
best_acc = 0.0

for epoch in range(1, PROBE_EPOCHS + 1 if True else FT_EPOCHS + 1):
    probe.train()
    probe.encoder.eval()
    tr_loss, tr_acc = train_one_epoch(probe, probe_train_loader, optimizer, criterion)
    te_loss, te_acc = evaluate(probe, probe_test_loader, criterion)
    train_losses.append(tr_loss); test_losses.append(te_loss)
    train_accs.append(tr_acc);   test_accs.append(te_acc)
    print(f"epoch {epoch:2d} | train {tr_loss:.4f} {tr_acc*100:.1f}% | test {te_loss:.4f} {te_acc*100:.1f}%")

    if te_acc > best_acc:
        best_acc = te_acc
        torch.save({'epoch': epoch, 'model': probe.state_dict(),
                    'test_accs': test_accs, 'train_accs': train_accs,
                    'test_losses': test_losses, 'train_losses': train_losses},
                   os.path.join(CKPT_DIR, 'dino_probe_best.pt'))

print(f"\nBest test accuracy : {max(test_accs)*100:.2f}%")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label='train'); ax1.plot(test_losses, label='test')
ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.legend(); ax1.set_title('Loss')
ax2.plot([a*100 for a in train_accs], label='train'); ax2.plot([a*100 for a in test_accs], label='test')
ax2.set_xlabel('epoch'); ax2.set_ylabel('acc (%)'); ax2.legend(); ax2.set_title('Accuracy')
plt.tight_layout(); plt.show()

---
## Full fine-tune

In [ ]:
class FullFineTune(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.head    = nn.Linear(D_MODEL, num_classes)

    def forward(self, x):
        h = self.encoder(x)
        return self.head(h)

In [ ]:
ft = FullFineTune(build_eval_encoder(), NUM_CLASSES).to(device)
print(f"trainable : {sum(p.numel() for p in ft.parameters()):,}")

In [ ]:
optimizer = torch.optim.AdamW(ft.parameters(), lr=0.0001, weight_decay=0.0 if False else 0.05)
criterion = nn.CrossEntropyLoss()

train_losses, test_losses = [], []
train_accs,   test_accs   = [], []
best_acc = 0.0

for epoch in range(1, PROBE_EPOCHS + 1 if False else FT_EPOCHS + 1):
    ft.train()
    tr_loss, tr_acc = train_one_epoch(ft, probe_train_loader, optimizer, criterion)
    te_loss, te_acc = evaluate(ft, probe_test_loader, criterion)
    train_losses.append(tr_loss); test_losses.append(te_loss)
    train_accs.append(tr_acc);   test_accs.append(te_acc)
    print(f"epoch {epoch:2d} | train {tr_loss:.4f} {tr_acc*100:.1f}% | test {te_loss:.4f} {te_acc*100:.1f}%")

    if te_acc > best_acc:
        best_acc = te_acc
        torch.save({'epoch': epoch, 'model': ft.state_dict(),
                    'test_accs': test_accs, 'train_accs': train_accs,
                    'test_losses': test_losses, 'train_losses': train_losses},
                   os.path.join(CKPT_DIR, 'dino_ft_best.pt'))

print(f"\nBest test accuracy : {max(test_accs)*100:.2f}%")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label='train'); ax1.plot(test_losses, label='test')
ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.legend(); ax1.set_title('Loss')
ax2.plot([a*100 for a in train_accs], label='train'); ax2.plot([a*100 for a in test_accs], label='test')
ax2.set_xlabel('epoch'); ax2.set_ylabel('acc (%)'); ax2.legend(); ax2.set_title('Accuracy')
plt.tight_layout(); plt.show()